In [1]:
import pandas as pd
import joblib

live_df = pd.read_csv("../data/cicids2017/live_flows.csv")
rf = joblib.load("../model/trained/random_forest.pkl")

trained_features = rf.feature_names_in_

print("Trained model expects:", len(trained_features), "features")
print("Live capture has:", len(live_df.columns), "columns")

print("\nSample of trained feature names:")
print(list(trained_features)[:10])
print("\nSample of live capture column names:")
print(list(live_df.columns)[:10])

Trained model expects: 84 features
Live capture has: 82 columns

Sample of trained feature names:
['Src Port', 'Dst Port', 'Protocol', 'Flow Duration', 'Total Fwd Packet', 'Total Bwd packets', 'Total Length of Fwd Packet', 'Total Length of Bwd Packet', 'Fwd Packet Length Max', 'Fwd Packet Length Min']

Sample of live capture column names:
['src_ip', 'dst_ip', 'src_port', 'dst_port', 'protocol', 'timestamp', 'flow_duration', 'flow_byts_s', 'flow_pkts_s', 'fwd_pkts_s']


In [2]:
print(list(trained_features))
print(list(live_df.columns))

['Src Port', 'Dst Port', 'Protocol', 'Flow Duration', 'Total Fwd Packet', 'Total Bwd packets', 'Total Length of Fwd Packet', 'Total Length of Bwd Packet', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd RST Flags', 'Bwd RST Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Packet Length Min', 'Packet Length Max', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag

In [3]:
column_mapping = {
    'src_port': 'Src Port', 'dst_port': 'Dst Port', 'protocol': 'Protocol',
    'flow_duration': 'Flow Duration', 'tot_fwd_pkts': 'Total Fwd Packet',
    'tot_bwd_pkts': 'Total Bwd packets', 'totlen_fwd_pkts': 'Total Length of Fwd Packet',
    'totlen_bwd_pkts': 'Total Length of Bwd Packet', 'fwd_pkt_len_max': 'Fwd Packet Length Max',
    'fwd_pkt_len_min': 'Fwd Packet Length Min', 'fwd_pkt_len_mean': 'Fwd Packet Length Mean',
    'fwd_pkt_len_std': 'Fwd Packet Length Std', 'bwd_pkt_len_max': 'Bwd Packet Length Max',
    'bwd_pkt_len_min': 'Bwd Packet Length Min', 'bwd_pkt_len_mean': 'Bwd Packet Length Mean',
    'bwd_pkt_len_std': 'Bwd Packet Length Std', 'flow_byts_s': 'Flow Bytes/s',
    'flow_pkts_s': 'Flow Packets/s', 'flow_iat_mean': 'Flow IAT Mean',
    'flow_iat_std': 'Flow IAT Std', 'flow_iat_max': 'Flow IAT Max', 'flow_iat_min': 'Flow IAT Min',
    'fwd_iat_tot': 'Fwd IAT Total', 'fwd_iat_mean': 'Fwd IAT Mean', 'fwd_iat_std': 'Fwd IAT Std',
    'fwd_iat_max': 'Fwd IAT Max', 'fwd_iat_min': 'Fwd IAT Min', 'bwd_iat_tot': 'Bwd IAT Total',
    'bwd_iat_mean': 'Bwd IAT Mean', 'bwd_iat_std': 'Bwd IAT Std', 'bwd_iat_max': 'Bwd IAT Max',
    'bwd_iat_min': 'Bwd IAT Min', 'fwd_psh_flags': 'Fwd PSH Flags', 'bwd_psh_flags': 'Bwd PSH Flags',
    'fwd_urg_flags': 'Fwd URG Flags', 'bwd_urg_flags': 'Bwd URG Flags',
    'fwd_header_len': 'Fwd Header Length', 'bwd_header_len': 'Bwd Header Length',
    'fwd_pkts_s': 'Fwd Packets/s', 'bwd_pkts_s': 'Bwd Packets/s', 'pkt_len_min': 'Packet Length Min',
    'pkt_len_max': 'Packet Length Max', 'pkt_len_mean': 'Packet Length Mean',
    'pkt_len_std': 'Packet Length Std', 'pkt_len_var': 'Packet Length Variance',
    'fin_flag_cnt': 'FIN Flag Count', 'syn_flag_cnt': 'SYN Flag Count',
    'rst_flag_cnt': 'RST Flag Count', 'psh_flag_cnt': 'PSH Flag Count',
    'ack_flag_cnt': 'ACK Flag Count', 'urg_flag_cnt': 'URG Flag Count',
    'cwr_flag_count': 'CWR Flag Count', 'ece_flag_cnt': 'ECE Flag Count',
    'down_up_ratio': 'Down/Up Ratio', 'pkt_size_avg': 'Average Packet Size',
    'fwd_seg_size_avg': 'Fwd Segment Size Avg', 'bwd_seg_size_avg': 'Bwd Segment Size Avg',
    'fwd_byts_b_avg': 'Fwd Bytes/Bulk Avg', 'fwd_pkts_b_avg': 'Fwd Packet/Bulk Avg',
    'fwd_blk_rate_avg': 'Fwd Bulk Rate Avg', 'bwd_byts_b_avg': 'Bwd Bytes/Bulk Avg',
    'bwd_pkts_b_avg': 'Bwd Packet/Bulk Avg', 'bwd_blk_rate_avg': 'Bwd Bulk Rate Avg',
    'subflow_fwd_pkts': 'Subflow Fwd Packets', 'subflow_fwd_byts': 'Subflow Fwd Bytes',
    'subflow_bwd_pkts': 'Subflow Bwd Packets', 'subflow_bwd_byts': 'Subflow Bwd Bytes',
    'init_fwd_win_byts': 'FWD Init Win Bytes', 'init_bwd_win_byts': 'Bwd Init Win Bytes',
    'fwd_act_data_pkts': 'Fwd Act Data Pkts', 'fwd_seg_size_min': 'Fwd Seg Size Min',
    'active_mean': 'Active Mean', 'active_std': 'Active Std', 'active_max': 'Active Max',
    'active_min': 'Active Min', 'idle_mean': 'Idle Mean', 'idle_std': 'Idle Std',
    'idle_max': 'Idle Max', 'idle_min': 'Idle Min',
}

In [4]:
mapped_df = live_df.rename(columns=column_mapping)

# Fill missing columns not produced by this library
mapped_df['Fwd RST Flags'] = 0
mapped_df['Bwd RST Flags'] = 0
mapped_df['ICMP Code'] = -1   # matches CICIDS convention for non-ICMP flows
mapped_df['ICMP Type'] = -1
mapped_df['Total TCP Flow Time'] = mapped_df['Flow Duration']  # reasonable proxy

# Reorder to match exactly what the model expects
mapped_df = mapped_df[trained_features]

print(mapped_df.shape)
print(mapped_df.head())

(21, 84)
   Src Port  Dst Port  Protocol  Flow Duration  Total Fwd Packet  \
0     45448       443         6      26.523535                19   
1     58976       443         6      41.649099                 6   
2     57884       443         6      20.233971                 5   
3      1716      1716        17      49.997969                12   
4     35272        53        17       0.005406                 2   

   Total Bwd packets  Total Length of Fwd Packet  Total Length of Bwd Packet  \
0                 16                        9526                        1429   
1                  5                         396                         393   
2                  2                        2802                         926   
3                  0                       18168                           0   
4                  1                         166                          83   

   Fwd Packet Length Max  Fwd Packet Length Min  ...  Active Std  Active Max  \
0                   1

In [5]:
predictions = rf.predict(mapped_df)
probabilities = rf.predict_proba(mapped_df)[:, 1]

results = pd.DataFrame({
    'src_ip': live_df['src_ip'],
    'dst_ip': live_df['dst_ip'],
    'dst_port': live_df['dst_port'],
    'prediction': predictions,
    'confidence': probabilities
})

print(results)

            src_ip           dst_ip  dst_port  prediction  confidence
0    10.14.145.238    20.184.175.12       443           0        0.13
1    10.14.145.238    13.107.253.43       443           0        0.08
2    10.14.145.238       20.111.1.3       443           0        0.11
3    10.14.145.238    10.14.145.234      1716           0        0.11
4    10.14.145.238    10.14.145.234        53           0        0.08
5     57.144.39.32    10.14.145.238     33972           0        0.14
6   142.251.39.202    10.14.145.238     38984           0        0.17
7    216.198.79.67    10.14.145.238     56748           0        0.09
8    10.14.145.238   172.67.214.176       443           0        0.07
9    10.14.145.238  142.251.157.119       443           0        0.09
10   10.14.145.238    10.14.145.234        53           0        0.08
11   10.14.145.238    34.149.66.154       443           0        0.10
12   10.14.145.238   142.251.39.206       443           0        0.13
13   10.14.145.238  

In [6]:
attack_df = pd.read_csv("../data/cicids2017/live_flows_attack.csv")
mapped_attack_df = attack_df.rename(columns=column_mapping)

mapped_attack_df['Fwd RST Flags'] = 0
mapped_attack_df['Bwd RST Flags'] = 0
mapped_attack_df['ICMP Code'] = -1
mapped_attack_df['ICMP Type'] = -1
mapped_attack_df['Total TCP Flow Time'] = mapped_attack_df['Flow Duration']

mapped_attack_df = mapped_attack_df[trained_features]

predictions = rf.predict(mapped_attack_df)
probabilities = rf.predict_proba(mapped_attack_df)[:, 1]

results = pd.DataFrame({
    'src_ip': attack_df['src_ip'],
    'dst_ip': attack_df['dst_ip'],
    'dst_port': attack_df['dst_port'],
    'prediction': predictions,
    'confidence': probabilities
})

print(results.sort_values('confidence', ascending=False))

            src_ip           dst_ip  dst_port  prediction  confidence
2    13.107.253.43    10.14.145.238     55922           0        0.19
15   10.14.145.238   108.139.200.26       443           0        0.16
23   10.14.145.238    160.79.104.10       443           0        0.15
6    10.14.145.238  105.112.127.141       443           0        0.15
12   10.14.145.238     57.144.39.32      5222           0        0.14
8    10.14.145.238   142.251.39.206       443           0        0.13
5    140.82.112.25    10.14.145.238     33450           0        0.11
0   142.251.216.42    10.14.145.238     56373           0        0.11
7    10.14.145.238    160.79.104.10       443           0        0.11
3    10.14.145.238    10.14.145.234      1716           0        0.11
19   10.14.145.238    34.149.66.154       443           0        0.11
18   10.14.145.238    160.79.104.10       443           0        0.11
9    140.82.114.22    10.14.145.238     36398           0        0.10
21   10.14.145.238  

In [7]:
print(attack_df[attack_df['dst_ip'] == '172.17.0.2'])
print(attack_df.shape)

Empty DataFrame
Columns: [src_ip, dst_ip, src_port, dst_port, protocol, timestamp, flow_duration, flow_byts_s, flow_pkts_s, fwd_pkts_s, bwd_pkts_s, tot_fwd_pkts, tot_bwd_pkts, totlen_fwd_pkts, totlen_bwd_pkts, fwd_pkt_len_max, fwd_pkt_len_min, fwd_pkt_len_mean, fwd_pkt_len_std, bwd_pkt_len_max, bwd_pkt_len_min, bwd_pkt_len_mean, bwd_pkt_len_std, pkt_len_max, pkt_len_min, pkt_len_mean, pkt_len_std, pkt_len_var, fwd_header_len, bwd_header_len, fwd_seg_size_min, fwd_act_data_pkts, flow_iat_mean, flow_iat_max, flow_iat_min, flow_iat_std, fwd_iat_tot, fwd_iat_max, fwd_iat_min, fwd_iat_mean, fwd_iat_std, bwd_iat_tot, bwd_iat_max, bwd_iat_min, bwd_iat_mean, bwd_iat_std, fwd_psh_flags, bwd_psh_flags, fwd_urg_flags, bwd_urg_flags, fin_flag_cnt, syn_flag_cnt, rst_flag_cnt, psh_flag_cnt, ack_flag_cnt, urg_flag_cnt, ece_flag_cnt, down_up_ratio, pkt_size_avg, init_fwd_win_byts, init_bwd_win_byts, active_max, active_min, active_mean, active_std, idle_max, idle_min, idle_mean, idle_std, fwd_byts_b_av

In [8]:
attack_df = pd.read_csv("../data/cicids2017/live_flows_attack3.csv")

mapped_attack_df = attack_df.rename(columns=column_mapping)
mapped_attack_df['Fwd RST Flags'] = 0
mapped_attack_df['Bwd RST Flags'] = 0
mapped_attack_df['ICMP Code'] = -1
mapped_attack_df['ICMP Type'] = -1
mapped_attack_df['Total TCP Flow Time'] = mapped_attack_df['Flow Duration']

mapped_attack_df = mapped_attack_df[trained_features]

predictions = rf.predict(mapped_attack_df)
probabilities = rf.predict_proba(mapped_attack_df)[:, 1]

results = pd.DataFrame({
    'src_ip': attack_df['src_ip'],
    'dst_ip': attack_df['dst_ip'],
    'dst_port': attack_df['dst_port'],
    'prediction': predictions,
    'confidence': probabilities
})

print(results['prediction'].value_counts())
print(results.sort_values('confidence', ascending=False).head(20))

prediction
0    102
Name: count, dtype: int64
        src_ip      dst_ip  dst_port  prediction  confidence
29  172.17.0.1  172.17.0.2       781           0        0.11
18  172.17.0.1  172.17.0.2        23           0        0.11
16  172.17.0.1  172.17.0.2       111           0        0.11
20  172.17.0.1  172.17.0.2       587           0        0.11
19  172.17.0.1  172.17.0.2       135           0        0.11
7   172.17.0.1  172.17.0.2        22           0        0.11
27  172.17.0.1  172.17.0.2       636           0        0.11
60  172.17.0.1  172.17.0.2       662           0        0.11
58  172.17.0.1  172.17.0.2       442           0        0.11
61  172.17.0.1  172.17.0.2       191           0        0.11
66  172.17.0.1  172.17.0.2       202           0        0.11
67  172.17.0.1  172.17.0.2       561           0        0.11
68  172.17.0.1  172.17.0.2       731           0        0.11
62  172.17.0.1  172.17.0.2       289           0        0.11
13  172.17.0.1  172.17.0.2        25   

In [9]:
# comparing both datasets to figure out scan timing and portscan rows
# Load original training data again if not in memory
df_orig = pd.read_csv("../data/cicids2017/friday.csv")
portscan_rows = df_orig[df_orig['Label'] == 'Portscan']

print("CICIDS Portscan — Flow Duration stats:")
print(portscan_rows['Flow Duration'].describe())

print("\nYour live scan — Flow Duration stats:")
print(mapped_attack_df['Flow Duration'].describe())

print("\nCICIDS Portscan — SYN Flag Count stats:")
print(portscan_rows['SYN Flag Count'].describe())

print("\nYour live scan — SYN Flag Count stats:")
print(mapped_attack_df['SYN Flag Count'].describe())

CICIDS Portscan — Flow Duration stats:
count    1.590660e+05
mean     4.377893e+03
std      3.267220e+05
min      1.000000e+00
25%      4.200000e+01
50%      4.700000e+01
75%      6.100000e+01
max      1.072538e+08
Name: Flow Duration, dtype: float64

Your live scan — Flow Duration stats:
count    102.000000
mean       0.029439
std        0.297276
min        0.000000
25%        0.000000
50%        0.000005
75%        0.000006
max        3.002344
Name: Flow Duration, dtype: float64

CICIDS Portscan — SYN Flag Count stats:
count    159066.000000
mean          1.007161
std           0.086306
min           0.000000
25%           1.000000
50%           1.000000
75%           1.000000
max           2.000000
Name: SYN Flag Count, dtype: float64

Your live scan — SYN Flag Count stats:
count    102.000000
mean       1.715686
std        0.722721
min        0.000000
25%        2.000000
50%        2.000000
75%        2.000000
max        3.000000
Name: SYN Flag Count, dtype: float64


In [10]:
# trying to fix the scan duration time by converting the time from seconds to microseconds before feeding it to the model
time_based_cols = [
    'Flow Duration', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
    'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min',
    'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min',
    'Active Mean', 'Active Std', 'Active Max', 'Active Min',
    'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min', 'Total TCP Flow Time'
]

mapped_attack_df_fixed = mapped_attack_df.copy()
mapped_attack_df_fixed[time_based_cols] = mapped_attack_df_fixed[time_based_cols] * 1_000_000

predictions_fixed = rf.predict(mapped_attack_df_fixed)
probabilities_fixed = rf.predict_proba(mapped_attack_df_fixed)[:, 1]

print(pd.Series(predictions_fixed).value_counts())
print(probabilities_fixed[:20])

0    102
Name: count, dtype: int64
[0.01 0.1  0.1  0.1  0.1  0.1  0.1  0.11 0.1  0.1  0.1  0.1  0.1  0.1
 0.09 0.1  0.11 0.1  0.11 0.11]


In [11]:
importances = pd.Series(rf.feature_importances_, index=trained_features).sort_values(ascending=False)
print(importances.head(15))

Total Length of Fwd Packet    0.147828
Fwd Packet Length Mean        0.115787
Subflow Fwd Bytes             0.089245
Fwd Packet Length Max         0.081029
Fwd Segment Size Avg          0.079543
RST Flag Count                0.063815
SYN Flag Count                0.037858
Total TCP Flow Time           0.029978
Packet Length Min             0.028330
ACK Flag Count                0.027158
Protocol                      0.018945
Fwd Packet Length Std         0.016658
Fwd Seg Size Min              0.014718
Fwd Packet Length Min         0.014192
PSH Flag Count                0.014127
dtype: float64


In [12]:
top_15_features = importances.head(15).index.tolist()

print("Your live scan — top-15 feature values (mean):")
print(mapped_attack_df[top_15_features].mean())

print("\nCICIDS Portscan — top-15 feature values (mean):")
print(portscan_rows[top_15_features].mean())

print("\nCICIDS BENIGN — top-15 feature values (mean, for comparison):")
benign_rows = df_orig[df_orig['Label'] == 'BENIGN']
print(benign_rows[top_15_features].mean())

Your live scan — top-15 feature values (mean):
Total Length of Fwd Packet    124.784314
Fwd Packet Length Mean         58.967320
Subflow Fwd Bytes             124.784314
Fwd Packet Length Max          58.980392
Fwd Segment Size Avg           58.967320
RST Flag Count                  0.950980
SYN Flag Count                  1.715686
Total TCP Flow Time             0.029439
Packet Length Min              56.274510
ACK Flag Count                  0.950980
Protocol                        6.107843
Fwd Packet Length Std           0.018486
Fwd Seg Size Min               20.000000
Fwd Packet Length Min          58.941176
PSH Flag Count                  0.000000
dtype: float64

CICIDS Portscan — top-15 feature values (mean):
Total Length of Fwd Packet       0.058781
Fwd Packet Length Mean           0.009846
Subflow Fwd Bytes                0.005344
Fwd Packet Length Max            0.058781
Fwd Segment Size Avg             0.009846
RST Flag Count                   0.999057
SYN Flag Count        

Total Length of Fwd Packet    6.865257e+02
Fwd Packet Length Mean        6.269221e+01
Subflow Fwd Bytes             3.160291e+01
Fwd Packet Length Max         2.370717e+02
Fwd Segment Size Avg          6.269221e+01
RST Flag Count                1.646300e-01
SYN Flag Count                6.446746e-01
Total TCP Flow Time           2.685336e+07
Packet Length Min             2.851974e+01
ACK Flag Count                2.584146e+01
Protocol                      1.318332e+01
Fwd Packet Length Std         6.718446e+01
Fwd Seg Size Min              1.341165e+01
Fwd Packet Length Min         2.898463e+01
PSH Flag Count                3.509267e+00
dtype: float64


In [13]:
def fix_header_inclusion(df):
    df = df.copy()
    
    # Per-packet header size estimates (guard divide-by-zero)
    fwd_hdr_per_pkt = df['Fwd Header Length'] / df['Total Fwd Packet'].replace(0, 1)
    bwd_hdr_per_pkt = df['Bwd Header Length'] / df['Total Bwd packets'].replace(0, 1)
    
    # Group 1: totals — subtract actual total header bytes for that direction
    df['Total Length of Fwd Packet'] = (df['Total Length of Fwd Packet'] - df['Fwd Header Length']).clip(lower=0)
    df['Total Length of Bwd Packet'] = (df['Total Length of Bwd Packet'] - df['Bwd Header Length']).clip(lower=0)
    df['Subflow Fwd Bytes'] = df['Total Length of Fwd Packet']
    df['Subflow Bwd Bytes'] = df['Total Length of Bwd Packet']
    
    # Group 2: per-packet stats — subtract per-packet header estimate
    df['Fwd Packet Length Max'] = (df['Fwd Packet Length Max'] - fwd_hdr_per_pkt).clip(lower=0)
    df['Fwd Packet Length Min'] = (df['Fwd Packet Length Min'] - fwd_hdr_per_pkt).clip(lower=0)
    df['Fwd Packet Length Mean'] = (df['Fwd Packet Length Mean'] - fwd_hdr_per_pkt).clip(lower=0)
    df['Bwd Packet Length Max'] = (df['Bwd Packet Length Max'] - bwd_hdr_per_pkt).clip(lower=0)
    df['Bwd Packet Length Min'] = (df['Bwd Packet Length Min'] - bwd_hdr_per_pkt).clip(lower=0)
    df['Bwd Packet Length Mean'] = (df['Bwd Packet Length Mean'] - bwd_hdr_per_pkt).clip(lower=0)
    
    avg_hdr_per_pkt = (fwd_hdr_per_pkt + bwd_hdr_per_pkt) / 2
    df['Packet Length Min'] = (df['Packet Length Min'] - avg_hdr_per_pkt).clip(lower=0)
    df['Packet Length Max'] = (df['Packet Length Max'] - avg_hdr_per_pkt).clip(lower=0)
    df['Packet Length Mean'] = (df['Packet Length Mean'] - avg_hdr_per_pkt).clip(lower=0)
    df['Average Packet Size'] = (df['Average Packet Size'] - avg_hdr_per_pkt).clip(lower=0)
    df['Fwd Segment Size Avg'] = df['Fwd Packet Length Mean']
    df['Bwd Segment Size Avg'] = df['Bwd Packet Length Mean']
    
    # Group 4: recompute rate using corrected totals
    total_bytes = df['Total Length of Fwd Packet'] + df['Total Length of Bwd Packet']
    duration_sec = df['Flow Duration'].replace(0, 1e-9)  # avoid div by zero
    df['Flow Bytes/s'] = total_bytes / duration_sec
    
    return df

mapped_attack_df_fixed3 = fix_header_inclusion(mapped_attack_df)

predictions_fixed3 = rf.predict(mapped_attack_df_fixed3)
probabilities_fixed3 = rf.predict_proba(mapped_attack_df_fixed3)[:, 1]

print(pd.Series(predictions_fixed3).value_counts())
print(probabilities_fixed3[:20])

0    102
Name: count, dtype: int64
[0.06 0.15 0.14 0.14 0.14 0.14 0.14 0.16 0.14 0.14 0.14 0.14 0.14 0.15
 0.14 0.14 0.15 0.15 0.16 0.15]


In [14]:
# Combine both fixes: time scaling + header-length correction
mapped_attack_df_combined = mapped_attack_df.copy()

# Fix 1: scale duration/timing columns from seconds to microseconds
time_based_cols = [
    'Flow Duration', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
    'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min',
    'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min',
    'Active Mean', 'Active Std', 'Active Max', 'Active Min',
    'Idle Mean', 'Idle Std', 'Idle Max', 'Idle Min', 'Total TCP Flow Time'
]
mapped_attack_df_combined[time_based_cols] = mapped_attack_df_combined[time_based_cols] * 1_000_000

# Fix 2: header-length correction
mapped_attack_df_combined = fix_header_inclusion(mapped_attack_df_combined)

predictions_combined = rf.predict(mapped_attack_df_combined)
probabilities_combined = rf.predict_proba(mapped_attack_df_combined)[:, 1]

print(pd.Series(predictions_combined).value_counts())
print(probabilities_combined[:20])

0    102
Name: count, dtype: int64
[0.01 0.15 0.13 0.13 0.13 0.13 0.13 0.16 0.13 0.13 0.13 0.13 0.13 0.15
 0.14 0.13 0.14 0.15 0.16 0.14]


In [15]:
iat_cols = ['Flow IAT Mean', 'Flow IAT Max', 'Flow IAT Min', 'Flow IAT Std']

print("Your live scan (combined-fixed) — IAT stats:")
print(mapped_attack_df_combined[iat_cols].mean())

print("\nCICIDS Portscan — IAT stats:")
print(portscan_rows[iat_cols].mean())

Your live scan (combined-fixed) — IAT stats:
Flow IAT Mean    7360.797302
Flow IAT Max     9818.352905
Flow IAT Min        0.000000
Flow IAT Std     4250.639772
dtype: float64

CICIDS Portscan — IAT stats:
Flow IAT Mean     596.796323
Flow IAT Max     3145.848453
Flow IAT Min       50.049351
Flow IAT Std     1179.928351
dtype: float64


In [16]:
# Look at a single representative scan flow vs what BENIGN/Portscan look like
sample_flow = mapped_attack_df_combined[top_15_features].iloc[0]
print("One live scan flow (top-15 features):")
print(sample_flow)

print("\nDifference from CICIDS Portscan mean:")
print(sample_flow - portscan_rows[top_15_features].mean())

print("\nDifference from CICIDS BENIGN mean:")
print(sample_flow - benign_rows[top_15_features].mean())

One live scan flow (top-15 features):
Total Length of Fwd Packet    9.700000e+02
Fwd Packet Length Mean        1.940000e+02
Subflow Fwd Bytes             9.700000e+02
Fwd Packet Length Max         1.940000e+02
Fwd Segment Size Avg          1.940000e+02
RST Flag Count                0.000000e+00
SYN Flag Count                0.000000e+00
Total TCP Flow Time           3.002344e+06
Packet Length Min             2.040000e+02
ACK Flag Count                0.000000e+00
Protocol                      1.700000e+01
Fwd Packet Length Std         0.000000e+00
Fwd Seg Size Min              2.000000e+01
Fwd Packet Length Min         1.940000e+02
PSH Flag Count                0.000000e+00
Name: 0, dtype: float64

Difference from CICIDS Portscan mean:
Total Length of Fwd Packet    9.699412e+02
Fwd Packet Length Mean        1.939902e+02
Subflow Fwd Bytes             9.699947e+02
Fwd Packet Length Max         1.939412e+02
Fwd Segment Size Avg          1.939902e+02
RST Flag Count               -9.990570e

In [17]:
scan_flows = mapped_attack_df_combined[
    (attack_df['dst_ip'].values == '172.17.0.2') & 
    (mapped_attack_df_combined['Protocol'] == 6)
]

print(f"Found {len(scan_flows)} actual scan flows")

sample_flow = scan_flows[top_15_features].iloc[0]
print("\nOne actual scan flow (top-15 features):")
print(sample_flow)

print("\nDifference from CICIDS Portscan mean:")
print(sample_flow - portscan_rows[top_15_features].mean())

print("\nDifference from CICIDS BENIGN mean:")
print(sample_flow - benign_rows[top_15_features].mean())

Found 87 actual scan flows

One actual scan flow (top-15 features):
Total Length of Fwd Packet    76.000000
Fwd Packet Length Mean        38.000000
Subflow Fwd Bytes             76.000000
Fwd Packet Length Max         38.000000
Fwd Segment Size Avg          38.000000
RST Flag Count                 1.000000
SYN Flag Count                 2.000000
Total TCP Flow Time           27.179718
Packet Length Min             34.000000
ACK Flag Count                 1.000000
Protocol                       6.000000
Fwd Packet Length Std          0.000000
Fwd Seg Size Min              20.000000
Fwd Packet Length Min         38.000000
PSH Flag Count                 0.000000
Name: 1, dtype: float64

Difference from CICIDS Portscan mean:
Total Length of Fwd Packet      75.941219
Fwd Packet Length Mean          37.990154
Subflow Fwd Bytes               75.994656
Fwd Packet Length Max           37.941219
Fwd Segment Size Avg            37.990154
RST Flag Count                   0.000943
SYN Flag Count   

In [18]:
# Filter to closed-port scan flows: SYN present, but no ACK/RST response
# (i.e. no reply came back — a filtered or unresponsive port)
closed_port_flows = scan_flows[
    (scan_flows['SYN Flag Count'] > 0) & 
    (scan_flows['ACK Flag Count'] == 0) & 
    (scan_flows['RST Flag Count'] == 0)
]

print(f"Found {len(closed_port_flows)} closed-port-style scan flows")

if len(closed_port_flows) > 0:
    print("\nClosed-port flows — top-15 feature means:")
    print(closed_port_flows[top_15_features].mean())
    
    print("\nDifference from CICIDS Portscan mean:")
    print(closed_port_flows[top_15_features].mean() - portscan_rows[top_15_features].mean())
    
    # Run these specific flows through the model
    preds_closed = rf.predict(closed_port_flows[trained_features])
    probs_closed = rf.predict_proba(closed_port_flows[trained_features])[:, 1]
    print("\nPredictions on closed-port flows:")
    print(pd.Series(preds_closed).value_counts())
    print(probs_closed)

Found 18 closed-port-style scan flows

Closed-port flows — top-15 feature means:
Total Length of Fwd Packet    76.0
Fwd Packet Length Mean        38.0
Subflow Fwd Bytes             76.0
Fwd Packet Length Max         38.0
Fwd Segment Size Avg          38.0
RST Flag Count                 0.0
SYN Flag Count                 2.0
Total TCP Flow Time            0.0
Packet Length Min             48.0
ACK Flag Count                 0.0
Protocol                       6.0
Fwd Packet Length Std          0.0
Fwd Seg Size Min              20.0
Fwd Packet Length Min         38.0
PSH Flag Count                 0.0
dtype: float64

Difference from CICIDS Portscan mean:
Total Length of Fwd Packet      75.941219
Fwd Packet Length Mean          37.990154
Subflow Fwd Bytes               75.994656
Fwd Packet Length Max           37.941219
Fwd Segment Size Avg            37.990154
RST Flag Count                  -0.999057
SYN Flag Count                   0.992839
Total TCP Flow Time          -3580.350685
Pack


Predictions on closed-port flows:
0    18
Name: count, dtype: int64
[0.12 0.11 0.12 0.12 0.12 0.12 0.12 0.12 0.12 0.12 0.12 0.11 0.12 0.12
 0.12 0.12 0.12 0.12]


In [19]:
# For flows where total packet length equals what a bare header-only
# packet would be (no real payload was ever sent), force payload-related
# features to match CICIDS's 0-payload convention directly, rather than
# relying on an incomplete header-length subtraction.

def fix_header_inclusion_v2(df, header_only_threshold=70):
    df = df.copy()
    
    # Identify packets that are essentially header-only (very small, no payload)
    is_header_only_fwd = df['Fwd Packet Length Mean'] <= header_only_threshold

    payload_cols_fwd = [
        'Total Length of Fwd Packet', 'Subflow Fwd Bytes',
        'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean',
        'Fwd Segment Size Avg'
    ]
    for col in payload_cols_fwd:
        df.loc[is_header_only_fwd, col] = 0

    return df

mapped_attack_df_v2 = fix_header_inclusion_v2(mapped_attack_df_combined)

predictions_v2 = rf.predict(mapped_attack_df_v2[trained_features])
probabilities_v2 = rf.predict_proba(mapped_attack_df_v2[trained_features])[:, 1]

print(pd.Series(predictions_v2).value_counts())
print(probabilities_v2[:20])

1    69
0    33
Name: count, dtype: int64
[0.01 0.72 0.71 0.71 0.7  0.7  0.7  0.7  0.7  0.7  0.71 0.71 0.71 0.71
 0.51 0.7  0.7  0.7  0.7  0.7 ]


In [20]:
# Check what Fwd/Bwd Header Length actually contains for our scan flows
print(scan_flows[['Fwd Header Length', 'Bwd Header Length', 'Total Fwd Packet', 'Total Bwd packets']].head(10))

# Compute per-packet header overhead implied by this column
scan_flows_copy = scan_flows.copy()
scan_flows_copy['fwd_hdr_per_pkt'] = scan_flows_copy['Fwd Header Length'] / scan_flows_copy['Total Fwd Packet'].replace(0, 1)
print("\nImplied per-packet Fwd header size:")
print(scan_flows_copy['fwd_hdr_per_pkt'].describe())

    Fwd Header Length  Bwd Header Length  Total Fwd Packet  Total Bwd packets
1                  40                 20                 2                  1
2                  40                 20                 2                  1
3                  40                 20                 2                  1
4                  40                 20                 2                  1
5                  40                 20                 2                  1
6                  40                 20                 2                  1
7                  40                 20                 2                  1
8                  40                 20                 2                  1
9                  40                 20                 2                  1
10                 40                 20                 2                  1

Implied per-packet Fwd header size:
count    87.0
mean     20.0
std       0.0
min      20.0
25%      20.0
50%      20.0
75%      20.0
max    

In [21]:
def fix_header_inclusion_v3(df):
    df = df.copy()
    
    IP_HEADER_BYTES = 20  # standard IPv4 header, no options
    
    fwd_pkt_count = df['Total Fwd Packet'].replace(0, 1)
    bwd_pkt_count = df['Total Bwd packets'].replace(0, 1)
    
    # Total header overhead per direction = reported TCP header + missing IP header
    fwd_total_hdr = df['Fwd Header Length'] + (IP_HEADER_BYTES * df['Total Fwd Packet'])
    bwd_total_hdr = df['Bwd Header Length'] + (IP_HEADER_BYTES * df['Total Bwd packets'])
    fwd_hdr_per_pkt = fwd_total_hdr / fwd_pkt_count
    bwd_hdr_per_pkt = bwd_total_hdr / bwd_pkt_count
    
    df['Total Length of Fwd Packet'] = (df['Total Length of Fwd Packet'] - fwd_total_hdr).clip(lower=0)
    df['Total Length of Bwd Packet'] = (df['Total Length of Bwd Packet'] - bwd_total_hdr).clip(lower=0)
    df['Subflow Fwd Bytes'] = df['Total Length of Fwd Packet']
    df['Subflow Bwd Bytes'] = df['Total Length of Bwd Packet']
    
    df['Fwd Packet Length Max'] = (df['Fwd Packet Length Max'] - fwd_hdr_per_pkt).clip(lower=0)
    df['Fwd Packet Length Min'] = (df['Fwd Packet Length Min'] - fwd_hdr_per_pkt).clip(lower=0)
    df['Fwd Packet Length Mean'] = (df['Fwd Packet Length Mean'] - fwd_hdr_per_pkt).clip(lower=0)
    df['Bwd Packet Length Max'] = (df['Bwd Packet Length Max'] - bwd_hdr_per_pkt).clip(lower=0)
    df['Bwd Packet Length Min'] = (df['Bwd Packet Length Min'] - bwd_hdr_per_pkt).clip(lower=0)
    df['Bwd Packet Length Mean'] = (df['Bwd Packet Length Mean'] - bwd_hdr_per_pkt).clip(lower=0)
    
    avg_hdr_per_pkt = (fwd_hdr_per_pkt + bwd_hdr_per_pkt) / 2
    df['Packet Length Min'] = (df['Packet Length Min'] - avg_hdr_per_pkt).clip(lower=0)
    df['Packet Length Max'] = (df['Packet Length Max'] - avg_hdr_per_pkt).clip(lower=0)
    df['Packet Length Mean'] = (df['Packet Length Mean'] - avg_hdr_per_pkt).clip(lower=0)
    df['Average Packet Size'] = (df['Average Packet Size'] - avg_hdr_per_pkt).clip(lower=0)
    df['Fwd Segment Size Avg'] = df['Fwd Packet Length Mean']
    df['Bwd Segment Size Avg'] = df['Bwd Packet Length Mean']
    
    total_bytes = df['Total Length of Fwd Packet'] + df['Total Length of Bwd Packet']
    duration_sec = df['Flow Duration'].replace(0, 1e-9)
    df['Flow Bytes/s'] = total_bytes / duration_sec

    return df

mapped_attack_df_v3 = fix_header_inclusion_v3(mapped_attack_df_combined)

predictions_v3 = rf.predict(mapped_attack_df_v3[trained_features])
probabilities_v3 = rf.predict_proba(mapped_attack_df_v3[trained_features])[:, 1]

print(pd.Series(predictions_v3).value_counts())
print(probabilities_v3[:20])

1    69
0    33
Name: count, dtype: int64
[0.   0.83 0.83 0.83 0.82 0.82 0.82 0.82 0.82 0.82 0.83 0.83 0.83 0.83
 0.63 0.82 0.82 0.82 0.82 0.82]


In [22]:

#  just looking  at which flows are still predicted BENIGN
still_benign_mask = predictions_v3 == 0
still_benign_flows = mapped_attack_df_v3[trained_features][still_benign_mask]

print(f"{still_benign_mask.sum()} flows still classified BENIGN")
print("\nTheir SYN/ACK/RST flag pattern:")
print(mapped_attack_df_v3.loc[still_benign_flows.index, ['SYN Flag Count', 'ACK Flag Count', 'RST Flag Count', 'Fwd Packet Length Mean']].value_counts())

33 flows still classified BENIGN

Their SYN/ACK/RST flag pattern:
SYN Flag Count  ACK Flag Count  RST Flag Count  Fwd Packet Length Mean
2               0               0               0.0                       18
0               2               2               0.0                       14
                0               0               154.0                      1
Name: count, dtype: int64


In [23]:
print(mapped_attack_df_v3.loc[still_benign_flows.index, 'Protocol'].value_counts())

# Check direction - are Group 2 flows going the opposite way (container -> host)?
group2_mask = (mapped_attack_df_v3['SYN Flag Count'] == 0) & (mapped_attack_df_v3['ACK Flag Count'] == 2) & (mapped_attack_df_v3['RST Flag Count'] == 2)
print(attack_df.loc[mapped_attack_df_v3[group2_mask].index, ['src_ip', 'dst_ip']].head(10))

Protocol
6     32
17     1
Name: count, dtype: int64
        src_ip      dst_ip
73  172.17.0.2  172.17.0.1
78  172.17.0.2  172.17.0.1
79  172.17.0.2  172.17.0.1
80  172.17.0.2  172.17.0.1
84  172.17.0.2  172.17.0.1
86  172.17.0.2  172.17.0.1
87  172.17.0.2  172.17.0.1
90  172.17.0.2  172.17.0.1
93  172.17.0.2  172.17.0.1
94  172.17.0.2  172.17.0.1


In [24]:
# verifying it there's an index mismatcn between scan_flows and mapped_attack_df_v3
group1_mask = (mapped_attack_df_v3['SYN Flag Count'] == 2) & (mapped_attack_df_v3['ACK Flag Count'] == 0) & (mapped_attack_df_v3['RST Flag Count'] == 0)
group1_flows = mapped_attack_df_v3[group1_mask]

print(group1_flows[top_15_features].head())
print("\nConfidence on these specific flows:")
print(rf.predict_proba(group1_flows[trained_features])[:, 1])

    Total Length of Fwd Packet  Fwd Packet Length Mean  Subflow Fwd Bytes  \
70                           0                     0.0                  0   
71                           0                     0.0                  0   
72                           0                     0.0                  0   
74                           0                     0.0                  0   
75                           0                     0.0                  0   

    Fwd Packet Length Max  Fwd Segment Size Avg  RST Flag Count  \
70                    0.0                   0.0               0   
71                    0.0                   0.0               0   
72                    0.0                   0.0               0   
74                    0.0                   0.0               0   
75                    0.0                   0.0               0   

    SYN Flag Count  Total TCP Flow Time  Packet Length Min  ACK Flag Count  \
70               2                  0.0               28

In [25]:
# comparing the flagged group's features to the unflagged group's features to isolate the more important ones
flagged_mask = predictions_v3 == 1
flagged_flows = mapped_attack_df_v3[trained_features][flagged_mask]

print("Flagged group — Total TCP Flow Time & Packet Length Min:")
print(flagged_flows[['Total TCP Flow Time', 'Packet Length Min']].describe())

print("\nUnflagged Group 1 — same features:")
print(group1_flows[['Total TCP Flow Time', 'Packet Length Min']].describe())

Flagged group — Total TCP Flow Time & Packet Length Min:
       Total TCP Flow Time  Packet Length Min
count            69.000000               69.0
mean              6.371650                0.0
std               3.669014                0.0
min               4.768372                0.0
25%               5.245209                0.0
50%               5.483627                0.0
75%               5.722046                0.0
max              27.179718                0.0

Unflagged Group 1 — same features:
       Total TCP Flow Time  Packet Length Min
count                 18.0               18.0
mean                   0.0               28.0
std                    0.0                0.0
min                    0.0               28.0
25%                    0.0               28.0
50%                    0.0               28.0
75%                    0.0               28.0
max                    0.0               28.0


In [26]:
print("Flagged group — Total Fwd/Bwd Packet counts:")
print(mapped_attack_df_v3.loc[flagged_flows.index, ['Total Fwd Packet', 'Total Bwd packets']].describe())

print("\nUnflagged Group 1 — Total Fwd/Bwd Packet counts:")
print(mapped_attack_df_v3.loc[group1_flows.index, ['Total Fwd Packet', 'Total Bwd packets']].describe())

Flagged group — Total Fwd/Bwd Packet counts:
       Total Fwd Packet  Total Bwd packets
count         69.000000               69.0
mean           2.014493                1.0
std            0.120386                0.0
min            2.000000                1.0
25%            2.000000                1.0
50%            2.000000                1.0
75%            2.000000                1.0
max            3.000000                1.0

Unflagged Group 1 — Total Fwd/Bwd Packet counts:
       Total Fwd Packet  Total Bwd packets
count              18.0               18.0
mean                2.0                0.0
std                 0.0                0.0
min                 2.0                0.0
25%                 2.0                0.0
50%                 2.0                0.0
75%                 2.0                0.0
max                 2.0                0.0
